# 56. PDF 가스 확인 후보의 정상 참조 수·무작위성 민감도
54번 고정 열변화 하위5% 확인 후보를 가까운 정상20/30/40개와 RF·ExtraTrees 무작위설정3개묶음으로 확인한다. seed목록과 이웃수를 실행 전 선언하고 최고값만 고르지 않는다. 같은 자료 반복시험으로 독립 생산 검증은 아니다.
각 설정마다 내부 분위수를 새로 계산한다. 이전47번앵커는 고정, 새 전문가기만 변경한다. 모든 설정의 전체/과거 평가, 실제예측과 내부선택을 저장한다. 원본54보고서 결과를 덮어쓰지 않는다.


In [ ]:
from pathlib import Path
import json,pandas as pd
root=Path.cwd()
if not (root/'data').exists():root=root.parent
tables=root/'reports/tables'
source=''.join(json.loads((root/'notebooks/54_rg3_pdf_thermal_confirm_gates.ipynb').read_text(encoding='utf-8'))['cells'][1]['source'])
configs=[('k20_seed42_7_101',20,[42,7,101]),('k30_seed42_7_101',30,[42,7,101]),('k40_seed42_7_101',40,[42,7,101]),('k30_seed11_23_37',30,[11,23,37]),('k30_seed13_29_53',30,[13,29,53])]
metrics_all=[];predictions_all=[];choices_all=[]
for tag,neighbor_count,seed_set in configs:
 modified=source.replace('k=min(31,len(normal))',f'k=min({neighbor_count+1},len(normal))').replace('indexes=indexes[indexes!=i][:30]',f'indexes=indexes[indexes!=i][:{neighbor_count}]').replace('for seed in [42,7,101]:',f'for seed in {seed_set}:')
 modified=modified.replace('rg3_pdf_confirm_gates_',f'rg3_pdf_sensitivity_{tag}_')
 # Separate namespace prevents cached predictions from another configuration.
 ns={'__name__':'__main__'}
 exec(modified,ns)
 for name,collection in [('metrics',metrics_all),('predictions',predictions_all),('choices',choices_all)]:
  frame=pd.read_csv(tables/f'rg3_pdf_sensitivity_{tag}_{name}.csv')
  frame['configuration']=tag;frame['neighbor_count']=neighbor_count;frame['seeds']=str(seed_set)
  collection.append(frame)
 print('Completed sensitivity configuration',tag,flush=True)
metrics=pd.concat(metrics_all,ignore_index=True)
predictions=pd.concat(predictions_all,ignore_index=True)
choices=pd.concat(choices_all,ignore_index=True)
metrics.to_csv(tables/'rg3_pdf_sensitivity_metrics.csv',index=False,encoding='utf-8-sig')
predictions.to_csv(tables/'rg3_pdf_sensitivity_predictions.csv',index=False,encoding='utf-8-sig')
choices.to_csv(tables/'rg3_pdf_sensitivity_choices.csv',index=False,encoding='utf-8-sig')
print('FIXED thermal gate lower5%: no selection of best configuration')
print(metrics[metrics.model.eq('thermal_change_low_0.05')].to_string(index=False))
print('TRAIN INNER GUARDED selector')
print(metrics[metrics.model.eq('inner_guarded_selector')].to_string(index=False))
